In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Combined Modular Classifier (Excl ESI 1): LightGBM (ESI 2 vs 3) & Undersampled XGBoost (ESI 4 vs 5) (`models/xgboost_esi23_esi45_extreme.ipynb`)

This notebook implements a **Modular Hierarchical Cascade Classifier** for non-ESI 1 patients combining **LightGBM** (trained strictly on natural ESI 2 vs 3), **XGBoost with Majority Downsampling** (trained strictly on ESI 4 vs 5), and an **XGBoost Branch Router** using **45 Predictor Features** (19 raw inputs from `config/triage_conf.json` + 10 binary vital anomaly flags + 16 continuous vital delta & range features), 5-Fold Stratified Cross-Validation, **Balanced Accuracy**, **Specificity**, and **MCC**:

### System Architecture & Resampling Strategy
1. **Global ESI 1 Row Filtering**: Removes all ESI 1 rows completely (`raw_esi != "1"`), focusing evaluation on ESI 2, 3, 4, and 5.
2. **Router Model (XGBoost ESI 2/3 vs ESI 4/5)**: Trained on natural un-sampled non-ESI 1 training partitions (`1` for ESI 2/3, `0` for ESI 4/5).
3. **Sub-Model 1: LightGBM ESI 2 vs 3 Specialist**:
   - Training Subset: ESI 2 and ESI 3 rows ONLY (ESI 4 & 5 strictly removed).
   - Resampling: **Natural un-sampled distribution** (no oversampling / no downsampling).
   - Binary Target: `1` for ESI 2, `0` for ESI 3.
   - Algorithm: LightGBM (`objective = "binary"`, `metric = "binary_logloss"`).
4. **Sub-Model 2: XGBoost ESI 4 vs 5 Specialist (WITH DOWNSAMPLING)**:
   - Training Subset: ESI 4 and ESI 5 rows ONLY (ESI 2 & 3 strictly removed).
   - Resampling: **Factor-Controlled Majority Downsampling** (`undersample_ratio = 1.0`) on training subset to balance ESI 4 vs ESI 5.
   - Binary Target: `1` for ESI 4, `0` for ESI 5.
   - Algorithm: XGBoost (`objective = "binary:logistic"`, `eval_metric = "logloss"`).
5. **Combined Joint Probability Inference (4 Classes: ESI 2, 3, 4, 5)**:
   - $P(\text{ESI 2}) = P_{\text{router}}(\text{ESI 2/3}) \times P_{\text{LGB}}(\text{ESI 2} \mid 2, 3)$
   - $P(\text{ESI 3}) = P_{\text{router}}(\text{ESI 2/3}) \times (1 - P_{\text{LGB}}(\text{ESI 2} \mid 2, 3))$
   - $P(\text{ESI 4}) = (1 - P_{\text{router}}(\text{ESI 2/3})) \times P_{\text{XGB}}(\text{ESI 4} \mid 4, 5)$
   - $P(\text{ESI 5}) = (1 - P_{\text{router}}(\text{ESI 2/3})) \times (1 - P_{\text{XGB}}(\text{ESI 4} \mid 4, 5))$
6. **Reports & Artifacts**:
   - **CSV Reports**: `reports/xgboost_esi23_esi45_5fold_cv_report.csv` and `reports/xgboost_esi23_esi45_test_report.csv`.
   - **Diagnostic Plots**: Metrics bar chart (`plots/xgboost_esi23_esi45_metrics_barchart.png`).
   - **Model Export**: Saved to `deploy/xgboost_esi23_esi45_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(xgboost)
})
has_lgb <- requireNamespace("lightgbm", quietly = TRUE)
if (has_lgb) {
  library(lightgbm)
  cat("LightGBM R package loaded for ESI 2 vs 3 Specialist.\n")
} else {
  cat("Note: LightGBM R package not installed. Using XGBoost binary fallback for ESI 2 vs 3 Specialist.\n")
}
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Filter Out ESI 1 Rows, Construct 45 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
# GLOBAL FILTERING: Remove ESI 1 rows completely
raw_esi_all <- as.character(raw_df[[target_col_name]])
keep_mask   <- raw_esi_all != "1"
raw_df  <- raw_df[keep_mask, ]
raw_esi <- raw_esi_all[keep_mask]
cat(sprintf("ESI 1 Filtering: Removed %d ESI 1 rows (Remaining rows: %d)\n", sum(!keep_mask), nrow(raw_df)))
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 45 Predictor Features
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0),
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
df_full$target_col <- factor(raw_esi, levels = c("2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Non-ESI 1 Dataset Ready: %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Natural Target Class Distribution (ESI 2, 3, 4, 5):\n")
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & 5-Fold Cross-Validation for Combined Modular Pipeline
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size # 0.15
# Partition into Train/Val Set (85%) and Holdout Test Set (15%)
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# DOWNSAMPLING FUNCTION FOR ESI 4 vs 5
undersample_ratio <- 1.0
undersample_binary <- function(df_sub, target_col = "target_col", ratio = 1.0) {
  counts <- table(df_sub[[target_col]])
  if (length(counts) < 2) return(df_sub)
  maj_cls <- names(counts)[which.max(counts)]
  min_cls <- names(counts)[which.min(counts)]
  min_cnt <- counts[[min_cls]]
  maj_cnt <- counts[[maj_cls]]
  target_maj_cnt <- round(min_cnt * ratio)
  if (target_maj_cnt < maj_cnt && target_maj_cnt > 0) {
    idx_maj <- which(df_sub[[target_col]] == maj_cls)
    idx_min <- which(df_sub[[target_col]] == min_cls)
    sampled_maj <- sample(idx_maj, size = target_maj_cnt, replace = FALSE)
    return(df_sub[sort(c(idx_min, sampled_maj)), ])
  }
  return(df_sub)
}
cat("=== Data Partitioning Summary (Excl ESI 1) ===\n")
cat(sprintf("Full Dataset (ESI 2,3,4,5): %d rows\n", nrow(df_full)))
cat(sprintf("Train/Val Set (85%%)       : %d rows\n", nrow(train_val_df)))
cat(sprintf("Holdout Test Set (15%%)     : %d rows\n\n", nrow(test_df)))
cat("Resampling Policy: No oversampling; Downsampling (1.0x ratio) applied ONLY to XGBoost ESI 4 vs 5 training set.\n\n")
k_folds <- 5
folds   <- createFolds(train_val_df$target_col, k = k_folds, list = TRUE, returnTrain = FALSE)
binary_cols  <- c("gender", "cc_breathingdifficulty",
                  "is_dyspnea_total", "is_dyspnea_moderate", "is_bradypnea", "is_tachypnea",
                  "is_hypotension", "is_hypertension", "is_bradycardia_total", "is_bradycardia_moderate",
                  "is_tachycardia_total", "is_tachycardia_moderate")
feature_cols <- setdiff(names(train_val_df), c(binary_cols, "target_col"))
cont_cols    <- feature_cols
val_fold_accs     <- numeric(k_folds)
val_fold_bal_accs <- numeric(k_folds)
val_fold_specs    <- numeric(k_folds)
cat("============================================================\n")
cat(sprintf("   STARTING %d-FOLD CV: MODULAR PIPELINE (NATURAL LGBM 2v3 + DOWNSAMPLED XGB 4v5 + ROUTER)\n", k_folds))
cat("============================================================\n")
for (k in 1:k_folds) {
  val_idx    <- folds[[k]]
  train_fold <- train_val_df[-val_idx, ]
  val_fold   <- train_val_df[val_idx, ]
  preproc_fold <- preProcess(train_fold[, cont_cols, drop = FALSE], method = c("center", "scale"))
  train_fold   <- predict(preproc_fold, train_fold)
  val_fold     <- predict(preproc_fold, val_fold)
  all_feats <- c(binary_cols, cont_cols)
  # 1. ROUTER MODEL: ESI 2/3 vs ESI 4/5 (Natural Distribution, No Oversampling)
  y_tr_router  <- ifelse(train_fold$target_col %in% c("2", "3"), 1, 0)
  y_val_router <- ifelse(val_fold$target_col %in% c("2", "3"), 1, 0)
  dtr_router <- xgb.DMatrix(data = as.matrix(train_fold[, all_feats]), label = y_tr_router)
  dvl_router <- xgb.DMatrix(data = as.matrix(val_fold[, all_feats]),   label = y_val_router)
  model_router <- xgb.train(
    params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
    data = dtr_router, nrounds = 150, watchlist = list(val = dvl_router), early_stopping_rounds = 20, verbose = 0
  )
  p_branch_23 <- predict(model_router, dvl_router)
  p_branch_45 <- 1 - p_branch_23
  # 2. SUB-MODEL 1: LightGBM ESI 2 vs ESI 3 (Filtered & NATURAL distribution, no oversampling)
  tr_23_subset <- train_fold[train_fold$target_col %in% c("2", "3"), ]
  x_tr_23 <- as.matrix(tr_23_subset[, all_feats])
  y_tr_23 <- ifelse(tr_23_subset$target_col == "2", 1, 0) # 1=ESI 2, 0=ESI 3
  val_x <- as.matrix(val_fold[, all_feats])
  if (has_lgb) {
    dtr_lgb23 <- lgb.Dataset(data = x_tr_23, label = y_tr_23)
    lgb_params <- list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1)
    model_23 <- lgb.train(params = lgb_params, data = dtr_lgb23, nrounds = 100, verbose = -1)
    p_esi2_given_23 <- predict(model_23, val_x)
  } else {
    dtr_xgb23 <- xgb.DMatrix(data = x_tr_23, label = y_tr_23)
    model_23  <- xgb.train(params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6), data = dtr_xgb23, nrounds = 100, verbose = 0)
    p_esi2_given_23 <- predict(model_23, xgb.DMatrix(data = val_x))
  }
  p_esi3_given_23 <- 1 - p_esi2_given_23
  # 3. SUB-MODEL 2: XGBoost ESI 4 vs ESI 5 (Filtered & WITH DOWNSAMPLING)
  tr_45_subset <- train_fold[train_fold$target_col %in% c("4", "5"), ]
  tr_45_subset <- undersample_binary(tr_45_subset, target_col = "target_col", ratio = undersample_ratio)
  x_tr_45 <- as.matrix(tr_45_subset[, all_feats])
  y_tr_45 <- ifelse(tr_45_subset$target_col == "4", 1, 0) # 1=ESI 4, 0=ESI 5
  dtr_xgb45 <- xgb.DMatrix(data = x_tr_45, label = y_tr_45)
  model_45  <- xgb.train(
    params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
    data = dtr_xgb45, nrounds = 100, verbose = 0
  )
  p_esi4_given_45 <- predict(model_45, xgb.DMatrix(data = val_x))
  p_esi5_given_45 <- 1 - p_esi4_given_45
  # 4. COMBINE PROBABILITIES FOR 4 CLASSES (ESI 2, 3, 4, 5)
  p_esi2 <- p_branch_23 * p_esi2_given_23
  p_esi3 <- p_branch_23 * p_esi3_given_23
  p_esi4 <- p_branch_45 * p_esi4_given_45
  p_esi5 <- p_branch_45 * p_esi5_given_45
  probs_mat <- cbind(p_esi2, p_esi3, p_esi4, p_esi5)
  colnames(probs_mat) <- c("2", "3", "4", "5")
  fold_pred_idx <- apply(probs_mat, 1, which.max)
  fold_pred_fac <- factor(colnames(probs_mat)[fold_pred_idx], levels = c("2", "3", "4", "5"))
  fold_cm       <- confusionMatrix(fold_pred_fac, val_fold$target_col)
  fold_acc      <- as.numeric(fold_cm$overall["Accuracy"])
  fold_bal_acc  <- mean(fold_cm$byClass[, "Balanced Accuracy"])
  fold_spec     <- mean(fold_cm$byClass[, "Specificity"])
  val_fold_accs[k]     <- fold_acc
  val_fold_bal_accs[k] <- fold_bal_acc
  val_fold_specs[k]    <- fold_spec
  cat(sprintf("  Validation Fold %d/%d Acc: %.4f | BalAcc: %.4f | Specificity: %.4f\n",
              k, k_folds, fold_acc, fold_bal_acc, fold_spec))
}
cat("============================================================\n")
cat(sprintf("   5-FOLD CV COMPLETE. Mean Acc = %.4f | Mean BalAcc = %.4f | Mean Spec = %.4f\n",
            mean(val_fold_accs), mean(val_fold_bal_accs), mean(val_fold_specs)))
cat("============================================================\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Final Production Modular Model Training & Holdout Test Benchmark
# ---------------------------------------------------------
cat("Training final production models on full train_val_df (85% data)...\n")
all_feats <- c(binary_cols, cont_cols)
preproc_tv <- preProcess(train_val_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_val_scaled <- predict(preproc_tv, train_val_df)
test_scaled      <- predict(preproc_tv, test_df)
test_x <- as.matrix(test_scaled[, all_feats])
# 1. Train Final Router Model (ESI 2/3 vs ESI 4/5, Natural Distribution)
y_tr_router  <- ifelse(train_val_scaled$target_col %in% c("2", "3"), 1, 0)
y_te_router  <- ifelse(test_scaled$target_col %in% c("2", "3"), 1, 0)
dtr_router <- xgb.DMatrix(data = as.matrix(train_val_scaled[, all_feats]), label = y_tr_router)
dte_router <- xgb.DMatrix(data = test_x,                                  label = y_te_router)
final_router <- xgb.train(
  params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
  data = dtr_router, nrounds = 150, watchlist = list(test = dte_router), early_stopping_rounds = 20, verbose = 0
)
test_p_branch_23 <- predict(final_router, dte_router)
test_p_branch_45 <- 1 - test_p_branch_23
# 2. Train Final LightGBM for ESI 2 vs 3 (Filter out ESI 4 & 5, Natural Distribution)
tr_23_final <- train_val_scaled[train_val_scaled$target_col %in% c("2", "3"), ]
x_tr_23 <- as.matrix(tr_23_final[, all_feats])
y_tr_23 <- ifelse(tr_23_final$target_col == "2", 1, 0)
if (has_lgb) {
  dtr_lgb23 <- lgb.Dataset(data = x_tr_23, label = y_tr_23)
  final_lgb_23 <- lgb.train(params = list(objective = "binary", metric = "binary_logloss", learning_rate = 0.05, num_leaves = 31, max_depth = 6, verbosity = -1), data = dtr_lgb23, nrounds = 100, verbose = -1)
  test_p_esi2_given_23 <- predict(final_lgb_23, test_x)
} else {
  dtr_xgb23 <- xgb.DMatrix(data = x_tr_23, label = y_tr_23)
  final_lgb_23 <- xgb.train(params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6), data = dtr_xgb23, nrounds = 100, verbose = 0)
  test_p_esi2_given_23 <- predict(final_lgb_23, dte_router)
}
test_p_esi3_given_23 <- 1 - test_p_esi2_given_23
# 3. Train Final XGBoost for ESI 4 vs 5 (Filter out ESI 2 & 3, WITH DOWNSAMPLING!)
tr_45_final <- train_val_scaled[train_val_scaled$target_col %in% c("4", "5"), ]
tr_45_final <- undersample_binary(tr_45_final, target_col = "target_col", ratio = undersample_ratio)
x_tr_45 <- as.matrix(tr_45_final[, all_feats])
y_tr_45 <- ifelse(tr_45_final$target_col == "4", 1, 0)
dtr_xgb45 <- xgb.DMatrix(data = x_tr_45, label = y_tr_45)
final_xgb_45 <- xgb.train(
  params = list(objective = "binary:logistic", eval_metric = "logloss", eta = 0.05, max_depth = 6, subsample = 0.8, colsample_bytree = 0.8),
  data = dtr_xgb45, nrounds = 100, verbose = 0
)
test_p_esi4_given_45 <- predict(final_xgb_45, dte_router)
test_p_esi5_given_45 <- 1 - test_p_esi4_given_45
# 4. Combine Final Probabilities for 4 Classes (ESI 2, 3, 4, 5)
p_esi2_final <- test_p_branch_23 * test_p_esi2_given_23
p_esi3_final <- test_p_branch_23 * test_p_esi3_given_23
p_esi4_final <- test_p_branch_45 * test_p_esi4_given_45
p_esi5_final <- test_p_branch_45 * test_p_esi5_given_45
raw_test_probs <- cbind(p_esi2_final, p_esi3_final, p_esi4_final, p_esi5_final)
colnames(raw_test_probs) <- c("2", "3", "4", "5")
test_pred_idx <- apply(raw_test_probs, 1, which.max)
test_pred_fac <- factor(colnames(raw_test_probs)[test_pred_idx], levels = c("2", "3", "4", "5"))
act_test_fac  <- factor(test_df$target_col, levels = c("2", "3", "4", "5"))
cm_test  <- confusionMatrix(test_pred_fac, act_test_fac)
acc_test <- as.numeric(cm_test$overall["Accuracy"])
prec_by_class    <- as.numeric(cm_test$byClass[, "Pos Pred Value"])
rec_by_class     <- as.numeric(cm_test$byClass[, "Sensitivity"])
spec_by_class    <- as.numeric(cm_test$byClass[, "Specificity"])
bal_acc_by_class <- as.numeric(cm_test$byClass[, "Balanced Accuracy"])
prec_by_class[is.na(prec_by_class)]       <- 0
rec_by_class[is.na(rec_by_class)]         <- 0
spec_by_class[is.na(spec_by_class)]       <- 0
bal_acc_by_class[is.na(bal_acc_by_class)] <- 0
f1_by_class <- ifelse((prec_by_class + rec_by_class) > 0, 
                      2 * (prec_by_class * rec_by_class) / (prec_by_class + rec_by_class), 0)
roc_auc_by_class <- sapply(1:4, function(i) {
  cls_name <- levels(act_test_fac)[i]
  act_bin  <- ifelse(act_test_fac == cls_name, 1, 0)
  r_obj    <- tryCatch(pROC::roc(act_bin, raw_test_probs[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
mcc_by_class <- sapply(1:4, function(i) {
  cls <- levels(act_test_fac)[i]
  tp  <- sum(test_pred_fac == cls & act_test_fac == cls)
  tn  <- sum(test_pred_fac != cls & act_test_fac != cls)
  fp  <- sum(test_pred_fac == cls & act_test_fac != cls)
  fn  <- sum(test_pred_fac != cls & act_test_fac == cls)
  
  num   <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
})
actual_counts <- as.numeric(table(act_test_fac))
pred_counts   <- as.numeric(table(test_pred_fac))
diff_vec      <- pred_counts - actual_counts
diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
test_report_df <- data.frame(
  Class             = levels(act_test_fac),
  Actual_Count      = actual_counts,
  Pred_Count        = pred_counts,
  Diff              = diff_str,
  Precision         = round(prec_by_class, 4),
  Recall_Sens       = round(rec_by_class, 4),
  Specificity       = round(spec_by_class, 4),
  Balanced_Accuracy = round(bal_acc_by_class, 4),
  F1_Score          = round(f1_by_class, 4),
  ROC_AUC           = round(roc_auc_by_class, 4),
  MCC_Score         = round(mcc_by_class, 4)
)
macro_prec    <- mean(prec_by_class)
macro_rec     <- mean(rec_by_class)
macro_spec    <- mean(spec_by_class)
macro_bal_acc <- mean(bal_acc_by_class)
macro_f1      <- mean(f1_by_class)
macro_roc_auc <- mean(roc_auc_by_class, na.rm = TRUE)
macro_mcc     <- mean(mcc_by_class)
cat(sprintf("============================================================\n"))
cat(sprintf("   MODULAR HIERARCHICAL CASCADE (EXCL ESI 1) - TEST BENCHMARK\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  5-Fold CV Mean Val Acc : %.4f (%.2f%%)\n", mean(val_fold_accs), mean(val_fold_accs) * 100))
cat(sprintf("  5-Fold CV Mean Bal Acc : %.4f (%.2f%%)\n", mean(val_fold_bal_accs), mean(val_fold_bal_accs) * 100))
cat(sprintf("  Final Test Accuracy    : %.4f (%.2f%%)\n", acc_test, acc_test * 100))
cat(sprintf("  Macro Precision        : %.4f\n", macro_prec))
cat(sprintf("  Macro Recall (Sens)    : %.4f\n", macro_rec))
cat(sprintf("  Macro Specificity      : %.4f\n", macro_spec))
cat(sprintf("  Macro Balanced Acc     : %.4f\n", macro_bal_acc))
cat(sprintf("  Macro F1-Score         : %.4f\n", macro_f1))
cat(sprintf("  Macro ROC-AUC          : %.4f\n", macro_roc_auc))
cat(sprintf("  Macro MCC Score        : %.4f\n", macro_mcc))
cat(sprintf("============================================================\n\n"))
cat("Holdout Test Set Per-Class Metrics (ESI 2, 3, 4, 5):\n")
print(test_report_df)
cat("\nHoldout Test Set 4x4 Confusion Matrix (Rows: Predicted, Columns: Actual):\n")
print(cm_test$table)
cat(sprintf("============================================================\n\n"))
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
cv_summary_df <- data.frame(
  Fold = paste0("Fold_", 1:k_folds),
  Validation_Accuracy = round(val_fold_accs, 4),
  Validation_Balanced_Accuracy = round(val_fold_bal_accs, 4),
  Validation_Specificity = round(val_fold_specs, 4)
)
write.csv(cv_summary_df,  file = file.path(reports_dir, "xgboost_esi23_esi45_5fold_cv_report.csv"), row.names = FALSE)
write.csv(test_report_df, file = file.path(reports_dir, "xgboost_esi23_esi45_test_report.csv"),     row.names = FALSE)
cat("5-Fold CV Validation CSV Report written to: reports/xgboost_esi23_esi45_5fold_cv_report.csv\n")
cat("Holdout Test Set CSV Report written to:       reports/xgboost_esi23_esi45_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Diagnostic Plots (Metrics Bar Chart including Specificity & MCC)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("5Fold_Val_Acc", "5Fold_Val_BalAcc", "Test_Accuracy", "Test_Macro_BalAcc", "Test_Macro_Prec", "Test_Macro_Rec", "Test_Macro_Spec", "Test_Macro_F1", "Test_Macro_AUC", "Test_Macro_MCC"),
                  levels = c("5Fold_Val_Acc", "5Fold_Val_BalAcc", "Test_Accuracy", "Test_Macro_BalAcc", "Test_Macro_Prec", "Test_Macro_Rec", "Test_Macro_Spec", "Test_Macro_F1", "Test_Macro_AUC", "Test_Macro_MCC")),
  Score  = c(mean(val_fold_accs), mean(val_fold_bal_accs), acc_test, macro_bal_acc, macro_prec, macro_rec, macro_spec, macro_f1, macro_roc_auc, macro_mcc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "Modular Hierarchical Cascade (Excl ESI 1: ESI 2, 3, 4, 5)",
       subtitle = "5-Fold CV vs Holdout Test (Natural LGBM 2v3 + Downsampled XGB 4v5 + Router)",
       y = "Metric Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")
ggsave(file.path(plots_dir, "xgboost_esi23_esi45_metrics_barchart.png"), plot = p_bar, width = 11, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_esi23_esi45_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Final Production Modular Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_esi23_esi45_extreme_model.rds")
saveRDS(list(router = final_router, model_lgb_23 = final_lgb_23, model_xgb_45 = final_xgb_45, preproc = preproc_tv, undersample_ratio = undersample_ratio), file = model_path)
cat("Final Production Modular Model Artifact saved to:", model_path, "\n")